In [ ]:
# Required imports
from typing import List, Dict, Any, Optional
from dataclasses import dataclass, field
from abc import ABC, abstractmethod
from enum import Enum
import re

print("✅ Imports successful")

## 1. What is Chain-of-Thought?

**Chain-of-Thought (CoT)** is a prompting technique where we ask the LLM to:
1. Break down complex problems into steps
2. Show its reasoning explicitly
3. Arrive at the answer step-by-step

This dramatically improves accuracy on complex reasoning tasks!

In [ ]:
# Without CoT (direct answer)
prompt_without_cot = """
Question: A store has 25 apples. They sell 8 apples in the morning and receive 
a shipment of 15 apples. How many apples do they have now?

Answer:
"""

# With CoT (step-by-step)
prompt_with_cot = """
Question: A store has 25 apples. They sell 8 apples in the morning and receive 
a shipment of 15 apples. How many apples do they have now?

Let's think step by step:
"""

print("Without CoT - LLM might answer: '32'")
print()
print("With CoT - LLM would answer:")
print("""
Let's think step by step:
1. Starting apples: 25
2. After selling 8: 25 - 8 = 17 apples
3. After receiving 15: 17 + 15 = 32 apples

Therefore, the store has 32 apples.
""")

## 2. Prompt Templates

Good CoT prompts have a structure:
1. **System instructions**: Tell the model HOW to think
2. **Examples** (optional): Show the desired format
3. **Question**: The actual problem
4. **Trigger phrase**: "Let's think step by step"

In [ ]:
@dataclass
class PromptTemplate:
    """A reusable prompt template with variable substitution."""
    template: str
    input_variables: List[str] = field(default_factory=list)
    
    def format(self, **kwargs) -> str:
        """Format the template with provided variables."""
        result = self.template
        for var in self.input_variables:
            if var not in kwargs:
                raise ValueError(f"Missing required variable: {var}")
            result = result.replace(f"{{{var}}}", str(kwargs[var]))
        return result

print("✅ PromptTemplate class defined")

In [ ]:
# Zero-shot CoT template (no examples)
ZERO_SHOT_COT = PromptTemplate(
    template="""
Question: {question}

Let's approach this step by step:

1. First, I'll identify the key information.
2. Then, I'll work through the problem logically.
3. Finally, I'll state my answer clearly.

Solution:
""",
    input_variables=["question"]
)

# Test it
question = "If a train travels at 60 mph for 2.5 hours, how far does it go?"
print(ZERO_SHOT_COT.format(question=question))

In [ ]:
# Few-shot CoT template (with examples)
FEW_SHOT_COT = PromptTemplate(
    template="""
I'll solve problems by showing my reasoning step by step.

Example 1:
Q: What is 15% of 80?
A: Let me work through this:
   Step 1: Convert 15% to a decimal: 15/100 = 0.15
   Step 2: Multiply: 0.15 × 80 = 12
   Answer: 12

Example 2:
Q: A car uses 8 gallons of gas for 240 miles. What is the mileage?
A: Let me work through this:
   Step 1: Mileage = miles / gallons
   Step 2: Mileage = 240 / 8 = 30
   Answer: 30 miles per gallon

Now solve this:
Q: {question}
A: Let me work through this:
""",
    input_variables=["question"]
)

# Test it
print(FEW_SHOT_COT.format(question="A shirt costs $40 and is 25% off. What's the sale price?"))

In [ ]:
# More template types

# Self-consistency template (for multiple answers)
SELF_CONSISTENCY = PromptTemplate(
    template="""
Question: {question}

I'll solve this problem {num_attempts} different ways to verify my answer.

Approach {attempt_num}:
""",
    input_variables=["question", "num_attempts", "attempt_num"]
)

# Step-back prompting (abstraction first)
STEP_BACK = PromptTemplate(
    template="""
Question: {question}

Before solving, let me identify the underlying principle:
Principle: {principle}

Now applying this principle:
""",
    input_variables=["question", "principle"]
)

# Decomposition template
DECOMPOSITION = PromptTemplate(
    template="""
Complex Question: {question}

Let me break this into sub-questions:
1. {sub_q1}
2. {sub_q2}
3. {sub_q3}

Solving each:
""",
    input_variables=["question", "sub_q1", "sub_q2", "sub_q3"]
)

print("✅ Multiple template types defined")

## 3. Reasoning Step Tracking

To build a reasoner, we need to track each step:

In [ ]:
@dataclass
class ReasoningStep:
    """A single step in the reasoning process."""
    step_number: int
    thought: str
    action: Optional[str] = None  # Tool to use
    action_input: Optional[Dict[str, Any]] = None  # Tool args
    observation: Optional[str] = None  # Result of action
    confidence: float = 1.0  # 0-1 confidence score
    
    def __str__(self) -> str:
        lines = [f"Step {self.step_number}: {self.thought}"]
        if self.action:
            lines.append(f"  → Action: {self.action}({self.action_input})")
        if self.observation:
            lines.append(f"  → Observation: {self.observation}")
        if self.confidence < 1.0:
            lines.append(f"  → Confidence: {self.confidence:.0%}")
        return "\n".join(lines)


@dataclass  
class ReasoningChain:
    """Complete chain of reasoning steps."""
    question: str
    steps: List[ReasoningStep] = field(default_factory=list)
    final_answer: Optional[str] = None
    
    def add_step(self, thought: str, **kwargs) -> ReasoningStep:
        """Add a new reasoning step."""
        step = ReasoningStep(
            step_number=len(self.steps) + 1,
            thought=thought,
            **kwargs
        )
        self.steps.append(step)
        return step
    
    def __str__(self) -> str:
        lines = [f"Question: {self.question}", "-" * 50]
        for step in self.steps:
            lines.append(str(step))
        if self.final_answer:
            lines.extend(["-" * 50, f"Answer: {self.final_answer}"])
        return "\n".join(lines)

print("✅ ReasoningStep and ReasoningChain defined")

In [ ]:
# Example: Manual reasoning chain

chain = ReasoningChain(question="What is 15% tip on a $45 meal?")

chain.add_step(
    thought="I need to calculate 15% of $45. First, I'll convert the percentage.",
    confidence=1.0
)

chain.add_step(
    thought="15% as a decimal is 0.15. Now I'll multiply by the bill amount.",
    action="calculator",
    action_input={"expression": "0.15 * 45"},
    observation="6.75"
)

chain.add_step(
    thought="The calculation shows $6.75. This is the tip amount.",
    confidence=0.99
)

chain.final_answer = "The 15% tip on a $45 meal is $6.75"

print(chain)

## 4. Building a Reasoner

A **Reasoner** generates reasoning chains. It can be:
- LLM-powered (using GPT, Claude, etc.)
- Algorithmic (rule-based)
- Hybrid

In [ ]:
class BaseReasoner(ABC):
    """Abstract base class for reasoners."""
    
    @abstractmethod
    def reason(self, question: str, **kwargs) -> ReasoningChain:
        """Generate a reasoning chain for the question."""
        pass
    
    def get_prompt(self, question: str) -> str:
        """Get the prompt for the LLM."""
        return ZERO_SHOT_COT.format(question=question)

print("✅ BaseReasoner defined")

In [ ]:
class SimpleMathReasoner(BaseReasoner):
    """
    A simple rule-based reasoner for math problems.
    This demonstrates the concept without needing an LLM.
    """
    
    def reason(self, question: str, **kwargs) -> ReasoningChain:
        chain = ReasoningChain(question=question)
        
        # Step 1: Parse the question
        chain.add_step(
            thought="Let me analyze this math problem and identify the numbers and operations."
        )
        
        # Step 2: Identify numbers
        numbers = re.findall(r'\d+\.?\d*', question)
        if numbers:
            chain.add_step(
                thought=f"I found these numbers: {', '.join(numbers)}"
            )
        
        # Step 3: Identify operation type
        operation = self._identify_operation(question)
        chain.add_step(
            thought=f"This appears to be a {operation} problem."
        )
        
        # Step 4: Calculate (simplified)
        if len(numbers) >= 2:
            nums = [float(n) for n in numbers]
            result = self._calculate(nums, operation)
            chain.add_step(
                thought=f"Performing the {operation} calculation.",
                action="calculator",
                action_input={"numbers": nums, "operation": operation},
                observation=str(result)
            )
            chain.final_answer = str(result)
        else:
            chain.final_answer = "Could not determine the answer."
        
        return chain
    
    def _identify_operation(self, question: str) -> str:
        """Identify the type of math operation."""
        q = question.lower()
        if any(w in q for w in ['add', 'sum', 'plus', 'total', '+']):
            return "addition"
        elif any(w in q for w in ['subtract', 'minus', 'difference', 'less', '-']):
            return "subtraction"
        elif any(w in q for w in ['multiply', 'times', 'product', '*', '×', 'of']):
            return "multiplication"
        elif any(w in q for w in ['divide', 'quotient', 'per', '/', '÷']):
            return "division"
        elif '%' in q or 'percent' in q:
            return "percentage"
        return "unknown"
    
    def _calculate(self, nums: List[float], operation: str) -> float:
        """Perform the calculation."""
        if operation == "addition":
            return sum(nums)
        elif operation == "subtraction":
            return nums[0] - sum(nums[1:])
        elif operation == "multiplication":
            result = 1
            for n in nums:
                result *= n
            return result
        elif operation == "division" and nums[1] != 0:
            return nums[0] / nums[1]
        elif operation == "percentage":
            return nums[0] * nums[1] / 100
        return 0

print("✅ SimpleMathReasoner defined")

In [ ]:
# Test the reasoner

reasoner = SimpleMathReasoner()

questions = [
    "What is 25 plus 17?",
    "Calculate 15% of 80",
    "If I have 100 and spend 35, how much is left?",
    "What is 12 times 8?"
]

for q in questions:
    print("\n" + "=" * 60)
    chain = reasoner.reason(q)
    print(chain)

## 5. Advanced: Tree-of-Thoughts

**Tree-of-Thoughts (ToT)** extends CoT by:
1. Generating multiple reasoning paths
2. Evaluating each path
3. Backtracking when needed
4. Selecting the best path

In [ ]:
@dataclass
class ThoughtNode:
    """A node in the thought tree."""
    thought: str
    score: float = 0.0  # Evaluation score
    children: List['ThoughtNode'] = field(default_factory=list)
    parent: Optional['ThoughtNode'] = None
    is_terminal: bool = False
    
    def add_child(self, thought: str, score: float = 0.0) -> 'ThoughtNode':
        """Add a child thought."""
        child = ThoughtNode(thought=thought, score=score, parent=self)
        self.children.append(child)
        return child
    
    def get_path(self) -> List['ThoughtNode']:
        """Get path from root to this node."""
        path = []
        node = self
        while node:
            path.append(node)
            node = node.parent
        return list(reversed(path))
    
    def __str__(self) -> str:
        return f"[{self.score:.2f}] {self.thought}"

print("✅ ThoughtNode defined")

In [ ]:
class TreeOfThoughtsReasoner:
    """
    Tree-of-Thoughts reasoner that explores multiple paths.
    """
    
    def __init__(self, branching_factor: int = 3, max_depth: int = 4):
        self.branching_factor = branching_factor
        self.max_depth = max_depth
    
    def reason(self, question: str) -> ThoughtNode:
        """Build a tree of thoughts and return the best path."""
        root = ThoughtNode(thought=f"Problem: {question}")
        
        # Expand the tree
        self._expand_node(root, depth=0)
        
        # Find best terminal node
        best_node = self._find_best_terminal(root)
        return best_node
    
    def _expand_node(self, node: ThoughtNode, depth: int) -> None:
        """Recursively expand a node."""
        if depth >= self.max_depth:
            node.is_terminal = True
            return
        
        # Generate child thoughts (in practice, would use LLM)
        thoughts = self._generate_thoughts(node, depth)
        
        for thought, score in thoughts:
            child = node.add_child(thought, score)
            
            # Only expand promising nodes
            if score > 0.5:
                self._expand_node(child, depth + 1)
            else:
                child.is_terminal = True
    
    def _generate_thoughts(self, node: ThoughtNode, depth: int) -> List[tuple]:
        """Generate possible next thoughts with scores."""
        # Simulated thought generation (in practice, would use LLM)
        import random
        
        thoughts = []
        prefixes = [
            ("Let me break this down: ", 0.8),
            ("One approach is to: ", 0.7),
            ("Alternatively, I could: ", 0.6),
            ("A simpler way might be: ", 0.75),
            ("This reminds me of: ", 0.5),
        ]
        
        selected = random.sample(prefixes, min(self.branching_factor, len(prefixes)))
        for prefix, base_score in selected:
            thought = f"{prefix}[Step {depth + 1} reasoning]"
            score = base_score * (0.8 + 0.4 * random.random())  # Add randomness
            thoughts.append((thought, min(score, 1.0)))
        
        return thoughts
    
    def _find_best_terminal(self, root: ThoughtNode) -> ThoughtNode:
        """Find the best terminal node."""
        best = None
        best_score = -1
        
        def search(node: ThoughtNode, path_score: float):
            nonlocal best, best_score
            
            current_score = path_score + node.score
            
            if node.is_terminal or not node.children:
                if current_score > best_score:
                    best = node
                    best_score = current_score
            else:
                for child in node.children:
                    search(child, current_score)
        
        search(root, 0)
        return best
    
    def print_tree(self, root: ThoughtNode, indent: int = 0) -> None:
        """Print the thought tree."""
        prefix = "  " * indent + ("├─ " if indent > 0 else "")
        print(f"{prefix}{root}")
        for child in root.children:
            self.print_tree(child, indent + 1)

print("✅ TreeOfThoughtsReasoner defined")

In [ ]:
# Test Tree-of-Thoughts

tot_reasoner = TreeOfThoughtsReasoner(branching_factor=2, max_depth=3)

question = "How can I improve my productivity?"
root = ThoughtNode(thought=f"Problem: {question}")

# Manually build a sample tree for visualization
approach1 = root.add_child("Focus on time management techniques", 0.85)
approach1.add_child("Use Pomodoro technique (25 min work, 5 min break)", 0.9)
approach1.add_child("Block calendar for deep work", 0.8)

approach2 = root.add_child("Optimize the environment", 0.75)
approach2.add_child("Remove distractions (phone, notifications)", 0.85)
approach2.add_child("Create a dedicated workspace", 0.7)

approach3 = root.add_child("Work on habits and mindset", 0.8)
approach3.add_child("Start with the hardest task (Eat the Frog)", 0.85)
approach3.add_child("Build consistent daily routines", 0.75)

print("🌳 Thought Tree:")
print("=" * 60)
tot_reasoner.print_tree(root)

# Find best path
best = tot_reasoner._find_best_terminal(root)
print("\n" + "=" * 60)
print("✅ Best reasoning path:")
for i, node in enumerate(best.get_path()):
    print(f"  {i}. {node}")

## 6. Self-Consistency

**Self-Consistency** generates multiple reasoning paths and takes a vote:

In [ ]:
from collections import Counter

class SelfConsistencyReasoner:
    """
    Generate multiple reasoning paths and vote on the answer.
    """
    
    def __init__(self, base_reasoner: BaseReasoner, num_samples: int = 5):
        self.base_reasoner = base_reasoner
        self.num_samples = num_samples
    
    def reason(self, question: str) -> Dict[str, Any]:
        """Generate multiple chains and vote."""
        chains = []
        answers = []
        
        # Generate multiple reasoning paths
        for i in range(self.num_samples):
            chain = self.base_reasoner.reason(question)
            chains.append(chain)
            if chain.final_answer:
                answers.append(chain.final_answer)
        
        # Vote on the answer
        if answers:
            vote_counts = Counter(answers)
            best_answer = vote_counts.most_common(1)[0]
            confidence = best_answer[1] / len(answers)
        else:
            best_answer = ("Unknown", 0)
            confidence = 0.0
        
        return {
            "question": question,
            "answer": best_answer[0],
            "votes": best_answer[1],
            "total_samples": len(answers),
            "confidence": confidence,
            "vote_distribution": dict(vote_counts) if answers else {},
            "chains": chains
        }

print("✅ SelfConsistencyReasoner defined")

In [ ]:
# Test self-consistency

sc_reasoner = SelfConsistencyReasoner(
    base_reasoner=SimpleMathReasoner(),
    num_samples=5
)

result = sc_reasoner.reason("What is 25 times 4?")

print("🗳️ Self-Consistency Results")
print("=" * 50)
print(f"Question: {result['question']}")
print(f"Final Answer: {result['answer']}")
print(f"Votes: {result['votes']} / {result['total_samples']}")
print(f"Confidence: {result['confidence']:.0%}")
print(f"Vote Distribution: {result['vote_distribution']}")

## 7. ReAct Pattern Preview

**ReAct (Reasoning + Acting)** combines thinking with tool use:

In [ ]:
REACT_TEMPLATE = PromptTemplate(
    template="""
You are a helpful assistant with access to these tools:
{tools}

Use this format:

Question: the input question
Thought: think about what to do
Action: tool_name
Action Input: {"arg": "value"}
Observation: result from the tool
... (repeat Thought/Action/Observation as needed)
Thought: I now know the final answer
Final Answer: the final answer

Begin!

Question: {question}
Thought:
""",
    input_variables=["tools", "question"]
)

# Example usage
tools_desc = """
- calculator: Evaluate math expressions. Input: {"expression": "2 + 2"}
- search: Search the web. Input: {"query": "search terms"}
"""

print(REACT_TEMPLATE.format(
    tools=tools_desc,
    question="What is the population of France?"
))

## Summary

In this notebook, we learned about Chain-of-Thought reasoning:

| Technique | Description |
|-----------|-------------|
| **Zero-shot CoT** | "Let's think step by step" (no examples) |
| **Few-shot CoT** | Provide examples of reasoning |
| **Self-Consistency** | Multiple paths + voting |
| **Tree-of-Thoughts** | Branching exploration with backtracking |
| **ReAct** | Interleaved reasoning and tool use |

### Key Components Built:
- `PromptTemplate`: Reusable prompt structures
- `ReasoningStep`: Individual reasoning step
- `ReasoningChain`: Complete chain of steps
- `BaseReasoner`: Abstract reasoner interface
- `ThoughtNode`: Tree structure for ToT
- `SelfConsistencyReasoner`: Multi-sample voting